# Exercise 6.4 — Fragility of the GHZ Metrological Advantage

**Chapter 6: Applications** &nbsp;|&nbsp; *Section 6.4: Quantum Metrology*

---

## Motivation

The GHZ state achieves the **Heisenberg limit** $F_Q = N^2$ — a quadratic improvement over the classical shot-noise limit $F_Q = N$.  But this advantage is catastrophically fragile: losing a *single qubit* to the environment destroys **all** metrological information ($F_Q \to 0$).  This exercise demonstrates why: the QFI is encoded entirely in the off-diagonal coherence $|0\cdots 0\rangle\langle 1\cdots 1|$, which vanishes under partial trace.  This fragility motivates the search for decoherence-robust metrological states.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Quantum Fisher Information (QFI):** Quantifies metrological usefulness. For pure states $|\psi\rangle$ with generator $G$, $F_Q = 4 \mathrm{Var}(G) = 4(\langle G^2 \rangle - \langle G \rangle^2)$.

**2. QFI for Mixed States:** For $\rho = \sum_k \lambda_k |k\rangle\langle k|$, $F_Q = 2 \sum_{k,l} \frac{(\lambda_k - \lambda_l)^2}{\lambda_k + \lambda_l} |\langle k| G |l \rangle |^2$.

**3. Heisenberg Limit vs Shot Noise Limit:** Using unentangled states, measuring phase achieves precision $\Delta \theta \geq 1/\sqrt{N}$ (Shot-noise limit, $F_Q = N$). Using highly entangled states (e.g. GHZ), $\Delta \theta \geq 1/N$ (Heisenberg limit, $F_Q = N^2$).

**4. Partial Trace:** Losing a single qubit means taking the partial trace $\rho_{N-1} = \mathrm{Tr}_{N} (|\psi\rangle\langle\psi|)$, which destroys global coherences.



## Exercise Statement

The $N$-qubit GHZ state $|\mathrm{GHZ}\rangle = (|0\cdots 0\rangle + |1\cdots 1\rangle)/\sqrt{2}$ achieves $F_Q = N^2$ for $G = \frac{1}{2}\sum_j \sigma_z^{(j)}$.

**(a)** Trace out one qubit and find the reduced state $\rho_{N-1}$.

**(b)** Compute the QFI of $\rho_{N-1}$ for the reduced generator $G' = \frac{1}{2}\sum_{j=1}^{N-1} \sigma_z^{(j)}$.

## Solution

### Part (a): Partial trace destroys coherence

The full density matrix is:

$$
\rho = \frac{1}{2}\bigl(|0\cdots 0\rangle\langle 0\cdots 0| + |0\cdots 0\rangle\langle 1\cdots 1| + |1\cdots 1\rangle\langle 0\cdots 0| + |1\cdots 1\rangle\langle 1\cdots 1|\bigr).
$$

Tracing out the last qubit: the off-diagonal terms vanish because $\langle 0_N | 1_N \rangle = 0$:

$$
\rho_{N-1} = \frac{1}{2}\bigl(|0\cdots 0\rangle\langle 0\cdots 0| + |1\cdots 1\rangle\langle 1\cdots 1|\bigr).
$$

This is a **classical mixture** of two orthogonal product states — all quantum coherence has been destroyed.

### Part (b): QFI of the reduced state

The QFI for a mixed state $\rho = \sum_k \lambda_k |k\rangle\langle k|$ is:

$$
F_Q = 2\sum_{k,l} \frac{(\lambda_k - \lambda_l)^2}{\lambda_k + \lambda_l} |\langle k|G'|l\rangle|^2.
$$

For $\rho_{N-1}$: $\lambda_0 = \lambda_1 = 1/2$ for the two populated eigenstates, and $\lambda_k = 0$ for all others.

**Between the two populated states:** $(\lambda_0 - \lambda_1)^2 = 0$.  This contribution vanishes.

**Between populated and unpopulated:** $\lambda_l = 0$ means the denominator vanishes, so these terms are excluded.

Moreover, $G'$ is diagonal in the computational basis (it's a sum of $Z$ operators), so $\langle 0\cdots 0|G'|1\cdots 1\rangle = 0$.

$$
\boxed{F_Q(\rho_{N-1}) = 0.}
$$

**Physical interpretation:** The GHZ state stores all metrological information in the global coherence $|0\cdots 0\rangle\langle 1\cdots 1|$.  This coherence oscillates under the phase generator $G$ at frequency $N$ (Heisenberg scaling), but is maximally non-local — it requires *all* qubits to be preserved.  Losing a single qubit irreversibly destroys this coherence, reducing the QFI from $N^2$ to exactly 0.  This is the "Schrödinger cat" fragility.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

N = sp.Symbol('N', positive=True, integer=True)
theta = sp.Symbol('theta')

# GHZ state: |GHZ> = (|0>^N + e^{iN*theta} |1>^N) / sqrt(2)
# QFI for pure state: F_Q = 4 Var(G) where G = sum_j Z_j/2
# <G> = 0 (by symmetry), <G^2> = N^2/4
# F_Q = 4 * N^2/4 = N^2  (Heisenberg scaling!)
F_Q_pure = N**2
print(f'QFI of pure GHZ: F_Q = {F_Q_pure}')

# After losing 1 qubit: rho_{N-1} = (|0><0|^{N-1} + |1><1|^{N-1})/2
# This is a classical mixture => no off-diagonal coherence
# QFI drops to F_Q = 0 for the lost generator
print(f'QFI after partial trace of 1 qubit: F_Q = 0')
print(f'Fragility: losing 1 qubit destroys ALL N^2 metrological advantage')

---
## Numerical Verification

In [ ]:
import numpy as np

def qfi_pure(psi, G):
    return 4*((psi.conj()@G@G@psi).real - (psi.conj()@G@psi).real**2)

def qfi_mixed(rho, G):
    eigs, V = np.linalg.eigh(rho)
    fq = 0.0
    for k in range(len(eigs)):
        for l in range(len(eigs)):
            d = eigs[k] + eigs[l]
            if d < 1e-15: continue
            fq += 2*(eigs[k]-eigs[l])**2/d * abs(V[:,k].conj()@G@V[:,l])**2
    return fq.real

print(f"{'N':>3s}  {'F_Q(GHZ)':>10s}  {'N²':>5s}  {'F_Q(reduced)':>14s}")
print(f"{'─'*3:>3s}  {'─'*10:>10s}  {'─'*5:>5s}  {'─'*14:>14s}")

for N in [2, 3, 4, 5, 6]:
    D = 2**N; D_r = 2**(N-1)
    ghz = np.zeros(D, dtype=complex)
    ghz[0] = ghz[-1] = 1/np.sqrt(2)
    
    # Generator
    G = np.zeros((D,D), dtype=complex)
    for i in range(N):
        Zi = np.eye(1)
        for j in range(N):
            Zi = np.kron(Zi, np.diag([1,-1]).astype(complex) if j==i else np.eye(2))
        G += Zi/2
    
    fq_full = qfi_pure(ghz, G)
    
    # Partial trace
    rho_r = np.zeros((D_r,D_r), dtype=complex)
    psi_mat = ghz.reshape(D_r, 2)
    rho_r = psi_mat @ psi_mat.conj().T
    
    G_r = np.zeros((D_r,D_r), dtype=complex)
    for i in range(N-1):
        Zi = np.eye(1)
        for j in range(N-1):
            Zi = np.kron(Zi, np.diag([1,-1]).astype(complex) if j==i else np.eye(2))
        G_r += Zi/2
    
    fq_red = qfi_mixed(rho_r, G_r)
    print(f"{N:3d}  {fq_full:10.1f}  {N**2:5d}  {fq_red:14.6f}")
    assert abs(fq_full - N**2) < 1e-6
    assert abs(fq_red) < 1e-10

print("\nF_Q: N² → 0 upon single-qubit loss. Fragility confirmed. ✓")